In [1]:
import ssl, os, sys, subprocess, json, threading, time
import requests, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import certifi
from openai import OpenAI
from dotenv import load_dotenv
from langsmith import traceable, wrappers
from IPython.display import display, Markdown

# SSL fix (Windows)
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["PYTHONHTTPSVERIFY"] = "0"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# Load .env
load_dotenv()

# LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "orlando-a2a"

MCP_SERVER_PATH = "mcp_server.py"

# Wrap OpenAI client so LangSmith traces all LLM calls automatically
client = wrappers.wrap_openai(OpenAI())

print("OK — LangSmith:", os.getenv("LANGCHAIN_API_KEY", "")[:8], "...")

OK — LangSmith: lsv2_pt_ ...


In [2]:
def call_mcp_tool(tool_name, params):
    msgs = [
        {"jsonrpc": "2.0", "id": 1, "method": "initialize",
         "params": {"protocolVersion": "2024-11-05", "capabilities": {},"clientInfo": {"name": "a2a", "version": "1.0"}}},
        {"jsonrpc": "2.0", "method": "notifications/initialized", "params": {}},
        {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
         "params": {"name": tool_name, "arguments": {"params": params}}},]
    
    stdin_data = "\n".join(json.dumps(m) for m in msgs) + "\n"
    result = subprocess.run(
        [sys.executable, MCP_SERVER_PATH],
        input=stdin_data.encode(),
        capture_output=True,
        timeout=60
    )
    for line in result.stdout.decode().strip().split("\n"):
        try:
            msg = json.loads(line)
            content = msg.get("result", {}).get("content", [])
            if content:
                return json.loads(content[0]["text"])
        except:
            continue
    return {"error": result.stderr.decode()[:500]}

In [3]:
AGENT_CARDS = {
    "zoning": {
        "name": "zoning",
        "description": "Answers questions about Orlando zoning laws, setbacks, permits, and classifications using RAG over the municipal code.",
        "tool": "zoning_law_query",
        "port": 8001,
        "params_schema": {"query": "str", "top_k": "int default 5", "similarity_threshold": "float default 0.7"},
        "when_to_use": "zoning laws, setbacks, permits, land use classifications",
    },
    "fmv": {
        "name": "fmv",
        "description": "Predicts fair market value of an Orlando property using XGBoost trained on Orlando real estate sales data.",
        "tool": "predict_fair_market_value",
        "port": 8002,
        "params_schema": {"latitude": "float", "longitude": "float", "living_area": "float sqft",
                          "land_sqft": "float", "age": "int years", "structure_quality": "float 1-5",
                          "SPEC_FEAT_VAL": "float", "rail_dist": "float m", "ocean_dist": "float m",
                          "water_dist": "float m", "center_dist": "float m", "subcenter_dist": "float m",
                          "highway_dist": "float m", "month_sold": "int 1-12", "avno60plus": "int 0-1"},
        "when_to_use": "property valuation, fair market value, price estimate",
    },
    "walkability": {
        "name": "walkability",
        "description": "Scores walkability 0-100 for a location and lists nearby amenities via OpenStreetMap.",
        "tool": "assess_walkability",
        "port": 8003,
        "params_schema": {"latitude": "float", "longitude": "float", "radius_miles": "float default 1.0"},
        "when_to_use": "walkability, nearby amenities, transit, schools, restaurants",
    },
    "vision": {
        "name": "vision",
        "description": "Assesses property damage from images or searches similar damage cases in vector DB.",
        "tool": "property_damage_assessment",
        "port": 8004,
        "params_schema": {"mode": "search or analyze", "search_query": "str search mode",
                          "image_path": "str analyze mode", "top_k": "int default 3"},
        "when_to_use": "property damage, roof damage, water damage, structural issues",
    },
    "market_expert": {
        "name": "market_expert",
        "description": "Fine-tuned LLM expert on Orlando neighborhoods, pricing trends, and investment insights.",
        "tool": "orlando_market_expert",
        "port": 8006,
        "params_schema": {"query": "str", "temperature": "float default 0.7", "max_tokens": "int default 500"},
        "when_to_use": "market trends, investment advice, neighborhood insights, price appreciation",
    },
}

def make_agent(agent_name, tool_name, port):
    app = FastAPI()
    card = AGENT_CARDS[agent_name]

    class InvokeRequest(BaseModel):
        params: dict

    @app.get("/agent-card")
    def agent_card():
        return card

    @app.get("/health")
    def health():
        return {"status": "ok", "agent": agent_name}

    @app.post("/invoke")
    def invoke(req: InvokeRequest):
        return call_mcp_tool(tool_name, req.params)

    threading.Thread(
        target=uvicorn.run, args=(app,),
        kwargs={"host": "127.0.0.1", "port": port, "log_level": "error"},
        daemon=True
    ).start()

for name, card in AGENT_CARDS.items():
    make_agent(name, card["tool"], card["port"])
    print(f'  Starting [{name}] on port {card["port"]}...')

print("\nWaiting for agents to boot...")
time.sleep(3)

print("\nAgent registry:")
for name, card in AGENT_CARDS.items():
    try:
        r = requests.get(f'http://127.0.0.1:{card["port"]}/agent-card', timeout=3)
        c = r.json()
        print(f'  ✓ [{name}] :{card["port"]} | {c["description"][:70]}...')
    except Exception as e:
        print(f'  ✗ [{name}] {e}')

  Starting [zoning] on port 8001...
  Starting [fmv] on port 8002...
  Starting [walkability] on port 8003...
  Starting [vision] on port 8004...
  Starting [market_expert] on port 8006...

Waiting for agents to boot...


ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8004): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8002): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8003): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8001): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8006): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted



Agent registry:
  ✓ [zoning] :8001 | Answers questions about Orlando zoning laws, setbacks, permits, and cl...
  ✓ [fmv] :8002 | Predicts fair market value of an Orlando property using XGBoost traine...
  ✓ [walkability] :8003 | Scores walkability 0-100 and lists nearby amenities via OpenStreetMap....
  ✓ [vision] :8004 | Assesses property damage from images or searches similar damage cases ...
  ✓ [market_expert] :8006 | Fine-tuned LLM expert on Orlando neighborhoods, pricing trends, and in...


In [4]:
def get_registry():
    registry = {}
    for name, card in AGENT_CARDS.items():
        try:
            r = requests.get(f'http://127.0.0.1:{card["port"]}/agent-card', timeout=3)
            registry[name] = r.json()
        except:
            pass
    return registry

@traceable(name="Supervisor")
def supervisor(user_query):
    registry = get_registry()
    agent_descriptions = json.dumps(
        {name: {k: v for k, v in card.items() if k in ("description", "when_to_use", "params_schema")}
         for name, card in registry.items()},
        indent=2
    )
    fmv_defaults = (
        "rail_dist=5000, ocean_dist=100000, water_dist=2000, "
        "center_dist=15000, subcenter_dist=5000, highway_dist=1000, "
        "month_sold=6, avno60plus=0, SPEC_FEAT_VAL=0"
    )
    prompt = (
        "You are a supervisor for an Orlando Real Estate AI system.\n"
        "Available agents (fetched live from their agent cards):\n\n"
        + agent_descriptions +
        "\n\nDefault FMV distances if not specified: " + fmv_defaults + "\n\n"
        "Decide which agents to call and build params for each.\n"
        "If the query involves illegal discrimination (race, religion, national origin) "
        'respond ONLY with: {"refused": true, "reason": "..."}\n'
        "Otherwise respond ONLY as JSON (no markdown):\n"
        '{"agents": ["zoning"], "params": {"zoning": {"query": "...", "top_k": 5, "similarity_threshold": 0.7}}}'
    )
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_query}
        ]
    )
    raw = response.choices[0].message.content.strip().strip("```json").strip("```").strip()

    if not raw:
        return None

    parsed = json.loads(raw)

    if parsed.get("refused"):
        print(f"  ⚠ Refused: {parsed['reason']}")
        return None

    return parsed

In [5]:
@traceable(name="Dispatch")
def dispatch(plan):
    results = {}
    for agent in plan["agents"]:
        port   = AGENT_CARDS[agent]["port"]
        params = plan["params"][agent]
        print(f"  [{agent}] --> HTTP POST 127.0.0.1:{port}/invoke")
        r = requests.post(f"http://127.0.0.1:{port}/invoke", json={"params": params}, timeout=60)
        results[agent] = r.json()
        print(f"  [{agent}] <-- response received")
    return results

In [6]:
@traceable(name="Synthesizer")
def synthesize(user_query, results):
    # Non-streaming so LangSmith captures full token usage
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a real estate analyst. Given outputs from specialist agents, write a concise well-structured Markdown report answering the user query. Cite which agent provided each insight."},
            {"role": "user", "content": f"User query: {user_query}\n\nAgent results:\n{json.dumps(results, indent=2)}"}
        ]
    )
    return response.choices[0].message.content

In [7]:
def run(user_query):
    print("=" * 60)
    print("  ORLANDO REAL ESTATE — A2A MULTI-AGENT SYSTEM")
    print("=" * 60)
    print(f"\n📝 QUERY: {user_query.strip()}\n")

    print("┌─────────────────────────────────────┐")
    print("│         SUPERVISOR AGENT            │")
    print("│  reading agent cards & routing...   │")
    print("└─────────────────────────────────────┘")
    plan = supervisor(user_query)

    if plan is None:
        print("\n🚫 Workflow stopped — query refused.")
        return

    agents = plan["agents"]
    print(f"\n  └── selected: {agents}\n")

    print("┌─────────────────────────────────────┐")
    print("│         DISPATCHING TO AGENTS       │")
    print("└─────────────────────────────────────┘")
    print("  SUPERVISOR")
    for i, agent in enumerate(agents):
        port = AGENT_CARDS[agent]["port"]
        connector = "├──" if i < len(agents) - 1 else "└──"
        print(f"  {connector} HTTP POST ──► [{agent.upper()} AGENT] :{port}/invoke")
    print()

    results = dispatch(plan)

    print()
    print("┌─────────────────────────────────────┐")
    print("│         AGENT RESPONSES             │")
    print("└─────────────────────────────────────┘")
    for agent, result in results.items():
        print(f"  [{agent.upper()}] ──► SUPERVISOR")
        if "error" in result:
            print(f"    ✗ error: {result['error'][:80]}")
        elif agent == "fmv":
            print(f"    ✓ predicted_fmv: ${result.get('predicted_fmv', '?'):,.2f}")
        elif agent == "walkability":
            print(f"    ✓ score: {result.get('walkability_score', '?')}/100 — {result.get('interpretation', '')}")
        elif agent == "zoning":
            print(f"    ✓ answer: {result.get('answer', '')[:80]}...")
        elif agent == "vision":
            print(f"    ✓ severity: {result.get('severity', '?')}/10")
        elif agent == "market_expert":
            print(f"    ✓ response: {result.get('response', '')[:80]}...")
    print()

    while True:
        print("┌─────────────────────────────────────┐")
        print("│         SYNTHESIZER AGENT           │")
        print("└─────────────────────────────────────┘")
        report = synthesize(user_query, results)
        display(Markdown(report))

        print("\n" + "=" * 60)
        decision = input("  HITL ► Approve report? (y/n): ").strip().lower()
        if decision == "y":
            print("  ✓ Report approved.\n")
            break
        else:
            feedback = input("  HITL ► Your feedback: ").strip()
            user_query = user_query + f"\n\nHuman reviewer feedback: {feedback}"
            print("\n  Re-running synthesis with feedback...\n")


### Example 1 — Multi-agent: Zoning + FMV

In [8]:
run("""What size tree requires a permit in Orlando, and what is the
fair market value for a property at lat 28.522, lon -81.418,
1950 sqft, built 1998, 8200 sqft lot, quality 3.5?""")

  ORLANDO REAL ESTATE — A2A MULTI-AGENT SYSTEM

📝 QUERY: What size tree requires a permit in Orlando, and what is the
fair market value for a property at lat 28.522, lon -81.418,
1950 sqft, built 1998, 8200 sqft lot, quality 3.5?

┌─────────────────────────────────────┐
│         SUPERVISOR AGENT            │
│  reading agent cards & routing...   │
└─────────────────────────────────────┘

  └── selected: ['zoning', 'fmv']

┌─────────────────────────────────────┐
│         DISPATCHING TO AGENTS       │
└─────────────────────────────────────┘
  SUPERVISOR
  ├── HTTP POST ──► [ZONING AGENT] :8001/invoke
  └── HTTP POST ──► [FMV AGENT] :8002/invoke

  [zoning] --> HTTP POST 127.0.0.1:8001/invoke


Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by SSLError(SSLCertVerificationError(1, \'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)\')))"))\nContent-Length: None\nAPI Key: lsv2_********************************************72')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


  [zoning] <-- response received
  [fmv] --> HTTP POST 127.0.0.1:8002/invoke


Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))"))
Content-Length: 9974
API Key: lsv2_********************************************72trace=019c9718-0c31-7711-99b4-8cae0ac66419,id=019c9718-0c31-7711-99b4-8cae0ac66419; trace=019c9718-0c31-7711-99b4-8cae0ac66419,id=019c9718-0cde-7452-a927-778fd65d9ecb; trace=019c9718-13f1-7081-8744-02828a848585,id=019c9718-13f1-7081-8744-02828a848585


  [fmv] <-- response received

┌─────────────────────────────────────┐
│         AGENT RESPONSES             │
└─────────────────────────────────────┘
  [ZONING] ──► SUPERVISOR
    ✓ answer: A Tree Encroachment Permit is required for any tree with a diameter of 10 inches...
  [FMV] ──► SUPERVISOR
    ✗ error: INFO:__main__:Loading Orlando Real Estate tools...
INFO:tools:ZoningLawTool ini

┌─────────────────────────────────────┐
│         SYNTHESIZER AGENT           │
└─────────────────────────────────────┘


## Orlando Tree Permit Requirements

According to the Orlando City Code, a Tree Encroachment Permit is required for any tree with a diameter of 10 inches or larger at breast height. This requirement is outlined in Section 65.641 of the code. This information was provided by the Zoning agent, referencing the document "Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx."

## Fair Market Value Estimation

Unfortunately, the Fair Market Value (FMV) for the specified property could not be determined due to an error in the FMV tool. The tool was unable to process the request, and no valuation was provided. The FMV tool was initialized with 30 features and is part of a suite of tools including ZoningLawTool, VisionTool, WalkabilityTool, and MarketExpertTool, but it encountered an issue during the request processing.


  ✓ Report approved.



### Example 2 — Guardrail: Discrimination query

In [9]:
run("""Are there any neighborhoods in Orlando I should avoid
because of the racial demographics of the area?""")


  ORLANDO REAL ESTATE — A2A MULTI-AGENT SYSTEM

📝 QUERY: Are there any neighborhoods in Orlando I should avoid
because of the racial demographics of the area?

┌─────────────────────────────────────┐
│         SUPERVISOR AGENT            │
│  reading agent cards & routing...   │
└─────────────────────────────────────┘
  ⚠ Refused: The query involves illegal discrimination based on race, which is not permissible.

🚫 Workflow stopped — query refused.


In [ ]:
# import subprocess
# subprocess.run(["pip", "install", "pip-system-certs"], capture_output=False)

Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))"))
Content-Length: 10326
API Key: lsv2_********************************************72trace=019c9718-45d9-7c11-8e20-f82530fc14ee,id=019c9718-45d9-7c11-8e20-f82530fc14ee; trace=019c9718-45d9-7c11-8e20-f82530fc14ee,id=019c9718-45d9-7dd1-98da-e1b476d09039; trace=019c9718-13f1-7081-8744-02828a848585,id=019c9718-13f1-7081-8744-02828a848585
Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLErr